In [1]:
from google import genai
import os
from dotenv import load_dotenv

import faiss
import numpy as np

In [2]:
#  SETUP GEMINI

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)


In [3]:

#  LOAD DOCUMENT

file_path = "usa.txt"

with open(file_path, "r", encoding="utf-8") as f:
    document = f.read()

print("Document loaded successfully!")
print("Total characters:", len(document))


Document loaded successfully!
Total characters: 6949


In [4]:

#  CHUNK DOCUMENT

chunk_size = 500
chunk_overlap = 50

chunks = []

start = 0

while start < len(document):

    end = start + chunk_size

    chunk = document[start:end]

    chunks.append(chunk)

    start = end - chunk_overlap


print("Total chunks:", len(chunks))



Total chunks: 16


In [5]:

#  CREATE EMBEDDINGS

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=chunks
)

chunk_embeddings = np.array(
    [embedding.values for embedding in result.embeddings],
    dtype="float32"
)

print("Embedding shape:", chunk_embeddings.shape)


Embedding shape: (16, 3072)


In [6]:

#  CREATE FAISS INDEX

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

print("Vectors stored in FAISS:", index.ntotal)


Vectors stored in FAISS: 16


In [7]:

# EMBED QUESTION

def embed_question(question):

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=[question]
    )

    question_embedding = np.array(
        [result.embeddings[0].values],
        dtype="float32"
    )

    return question_embedding


In [8]:

# SEARCH FAISS

def search_faiss(question_embedding, k=3):

    distances, indices = index.search(
        question_embedding,
        k=k
    )

    return indices[0]



In [9]:

#  RETRIEVE CHUNKS

def retrieve_chunks(indices):

    retrieved_chunks = []

    for i in indices:

        retrieved_chunks.append(chunks[i])

    return retrieved_chunks



In [10]:

#  CREATE CONTEXT

def create_context(retrieved_chunks):

    context = "\n\n".join(retrieved_chunks)

    return context


In [11]:
#  GENERATE ANSWER

def generate_answer(question, context):

    prompt = f"""
You are a helpful assistant.

Answer the question ONLY using the context below.

If the answer is not available in the context,
say:

"I don't know based on the provided document."

CONTEXT:
{context}

QUESTION:
{question}

Give a concise and accurate answer.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text



In [12]:
# COMPLETE RAG PIPELINE

def answer_question(question):

    # Question → Embedding
    question_embedding = embed_question(question)

    # Embedding → FAISS
    indices = search_faiss(
        question_embedding,
        k=3
    )

    # FAISS → Chunks
    retrieved_chunks = retrieve_chunks(indices)

    # Chunks → Context
    context = create_context(retrieved_chunks)

    # Context → Gemini
    answer = generate_answer(
        question,
        context
    )

    return answer


In [13]:
# QUESTION 1

question = "What are the major challenges facing the US economy?"

answer = answer_question(question)

print(answer)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The major challenges facing the US economy include rising debt, inequality (including a disproportionate share of wealth, access to affordable healthcare, housing shortages, and declining labor force participation in some sectors), political gridlock/polarization, and global competition (especially from China in areas like AI and semiconductors).


In [ ]:
#  QUESTION 2

question = "What is the GDP of USA in 2024?"

answer = answer_question(question)

print(answer)
